# Session 1: Data pipelines and core principles

**PGE 2 · Machine Learning · Dr. Souheil Hanoune**

80 minutes of coding · Dataset: **adult.csv** (Adult income, 1994 US census, UCI / OpenML, CC BY 4.0)

**How this notebook works**
- **Core** parts: everyone finishes them. **Extension** parts: only if you have time.
- Replace every `...` and every `raise NotImplementedError` with your code.
- Each function is followed by a **✅ Check** cell. Run it: no error means you can move on.
- Put this notebook and the CSV file(s) from Blackboard in the same folder (in Colab: the folder icon on the left, then upload).

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyClassifier
from sklearn.metrics import accuracy_score

RANDOM_STATE = 42
TARGET = "class"

def load_adult():
    """Load adult.csv from the current folder, or download it if the file is missing."""
    try:
        df = pd.read_csv("adult.csv")
    except FileNotFoundError:
        from sklearn.datasets import fetch_openml
        df = fetch_openml("adult", version=2, as_frame=True).frame
    for c in df.columns:
        if not pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].astype(object).map(lambda v: v.strip() if isinstance(v, str) else v)
    return df.replace("?", np.nan)

## Part 1 · Load and inspect (core)

**1.1** Load the data with `load_adult()`, then look at its shape, its column types and its first rows.

In [ ]:
df = ...          # load the data
# print the shape, the dtypes and the first 5 rows

In [ ]:
# ✅ Check
assert df.shape == (48842, 15), df.shape
print("OK")

**1.2** Which column is the target `y`? Which columns are features `X`? Is there a column that should **not** be used as a feature?

✍️ *Your answer:*

## Part 2 · Explore (core)

**2.1** Plot the histograms of `age` and `hours-per-week`, and print `describe()` for the numeric columns.

In [ ]:
# your code

**2.2** Look at `capital-gain`. What share of people have 0? What is the maximum, and how many people have exactly that value? Is it an outlier, a cap or a sentinel value?

In [ ]:
share_zero = ...      # share of rows with capital-gain == 0 (a number between 0 and 1)
n_max = ...           # number of rows equal to the maximum capital-gain
print(share_zero, df["capital-gain"].max(), n_max)

In [ ]:
# ✅ Check
assert 0.85 < share_zero < 0.95
assert n_max > 50
print("OK")

**2.2 bis** Outlier, cap or sentinel? Why?

✍️ *Your answer:*

**2.3** What share of people earn more than 50K? Then plot the share of `>50K` for each value of `education-num` (bar plot).

In [ ]:
rate_rich = ...       # share of rows with class ">50K"
# bar plot of the share of >50K by education-num

In [ ]:
# ✅ Check
assert 0.2 < rate_rich < 0.3
print("OK")

## Part 3 · Clean (core)

**3.1** Count the missing values per column and the number of duplicated rows.

In [ ]:
missing = ...         # a Series: number of missing values per column
n_dup = ...           # number of duplicated rows
print(missing[missing > 0]); print("duplicates:", n_dup)

In [ ]:
# ✅ Check
assert set(missing[missing > 0].index) == {"workclass", "occupation", "native-country"}
print("OK")

**3.2** Clean the data:
- drop the duplicated rows **first** (identical on every column, `fnlwgt` included: same record entered twice),
- then drop `fnlwgt` (a survey weight, not a property of the person) and `education` (same information as `education-num`).

After dropping `fnlwgt`, some rows look identical again: those are different people with the same profile, so we keep them.

Do **not** impute the missing values yet: imputation learns something (the most frequent value), so it must happen **after** the split.

In [ ]:
df_clean = ...

In [ ]:
# ✅ Check
EXPECTED = ["age", "workclass", "education-num", "marital-status", "occupation", "relationship",
            "race", "sex", "capital-gain", "capital-loss", "hours-per-week", "native-country", "class"]
assert list(df_clean.columns) == EXPECTED, list(df_clean.columns)
assert len(df_clean) == len(df) - n_dup
print("OK")

## Part 4 · Split (core)

Build `X` (all columns except the target) and `y` (1 if `>50K`, else 0). Then split **60 / 20 / 20** into train, validation and test, with `stratify` and `random_state=RANDOM_STATE`.

*Hint:* two calls to `train_test_split`: first 60 / 40, then split the 40 into two halves.

In [ ]:
X = ...
y = ...
X_train, X_tmp, y_train, y_tmp = ...
X_val, X_test, y_val, y_test = ...

In [ ]:
# ✅ Check
assert abs(len(X_train) / len(X) - 0.6) < 0.01
assert abs(y_train.mean() - y_val.mean()) < 0.01 and abs(y_train.mean() - y_test.mean()) < 0.01
print("OK: same share of >50K in the three sets")

## Part 5 · Prepare by hand (core)

You write the preparation in two functions:
- `fit_prepare(X_train)` **learns** what it needs **from the training set only**: the most frequent value of each categorical column, the mean and standard deviation of each numeric column, and the list of columns after one-hot encoding.
- `apply_prepare(X, stats)` **applies** it to any set: impute, one-hot encode, align the columns on the training columns, standardise.

Use `ddof=0` for the standard deviation (that is what scikit-learn does).

In [ ]:
NUM = ["age", "education-num", "capital-gain", "capital-loss", "hours-per-week"]
CAT = ["workclass", "marital-status", "occupation", "relationship", "race", "sex", "native-country"]

In [ ]:
def fit_prepare(X_train):
    """Return a dict with keys "modes", "means", "stds", "columns"."""
    raise NotImplementedError

def apply_prepare(X, stats):
    """Return a fully numeric DataFrame with exactly the columns stats["columns"]."""
    raise NotImplementedError

In [ ]:
stats = ...
X_train_p = ...
X_val_p = ...
X_test_p = ...

In [ ]:
# ✅ Check
for part in (X_train_p, X_val_p, X_test_p):
    assert not part.isna().any().any(), "missing values left"
    assert list(part.columns) == list(X_train_p.columns), "columns differ between sets"
assert np.allclose(X_train_p[NUM].mean(), 0, atol=1e-8) and np.allclose(X_train_p[NUM].std(ddof=0), 1)
print("OK")

**5.4** Why are the means of `X_val_p[NUM]` not exactly 0? Is that a problem?

✍️ *Your answer:*

## Part 6 · Rebuild it with scikit-learn (core)

Build the same preparation with a `ColumnTransformer`:
- numeric columns: `StandardScaler`,
- categorical columns: a `Pipeline` of `SimpleImputer(strategy="most_frequent")` then `OneHotEncoder(handle_unknown="ignore")`.

Fit it on `X_train` only, transform the three sets, and compare with Part 5.

In [ ]:
prep = ...

In [ ]:
# ✅ Check: same number of columns and same numeric block as your hand-made version
Xtr_sk = prep.transform(X_train)
Xtr_sk = Xtr_sk.toarray() if hasattr(Xtr_sk, "toarray") else Xtr_sk
assert Xtr_sk.shape == X_train_p.shape, (Xtr_sk.shape, X_train_p.shape)
assert np.allclose(Xtr_sk[:, :len(NUM)], X_train_p[NUM].to_numpy())
print("OK: both preparations match")

## Part 7 · Baseline (core if time, otherwise extension)

Fit a `DummyClassifier(strategy="most_frequent")` on the training set and compute its accuracy on the validation set.

In [ ]:
baseline_acc = ...

In [ ]:
# ✅ Check
assert 0.7 < baseline_acc < 0.8
print("OK")

**7.2** A model that always says "≤ 50K" gets this accuracy. Is it a good model? What would you need to measure instead?

✍️ *Your answer:*

## Extension

**E.1** Instead of imputing the most frequent value, replace missing categories by an explicit `"Unknown"` category. How many columns do you get after one-hot encoding? Which choice do you prefer, and why?

**E.2** Write a single function `prepare(df)` that goes from the raw file to `X_train_p, X_val_p, X_test_p, y_train, y_val, y_test`, and test it on a fresh `load_adult()`.

In [ ]:
# E.1

In [ ]:
# E.2